# 🔬 Benchmark Académico: Evaluación Comparativa de LLMs Locales en Quechua Collao
**Paper**: *Morpho-Semantic Scaffolding vs. Naive Parameter Scaling in Low-Resource Polysynthetic Languages (Quechua)*

Este notebook ejecuta la evaluación empírica de traducción Quechua -> Español para múltiples modelos abiertos servidos localmente en GPU mediante **Ollama**.

### ⚙️ Instrucciones Previas:
1. Ve a **Entorno de ejecución > Cambiar tipo de entorno de ejecución**.
2. Selecciona **GPU** (T4, L4 o A100 según tu suscripción Colab Pro/Plus).
3. Ejecuta las celdas en orden secuencial.

In [ ]:
# 1. Verificar aceleración GPU disponible
!nvidia-smi

In [ ]:
# 2. Clonar el repositorio e instalar dependencias
import os
repo_url = "https://github.com/RodrigoAlexander7/generative-quechua-translator.git"
if not os.path.exists("generative-quechua-translator"):
    !git clone {repo_url}
%cd generative-quechua-translator
!git pull origin main
!pip install -q tabulate sacrebleu pandas pytest


In [ ]:
# 3. Instalar y levantar Ollama con soporte CUDA en segundo plano
import subprocess, time, urllib.request

# Ollama requiere zstd para descomprimir su instalador
!apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

# Iniciar el daemon de Ollama (log en /content/ollama.log)
log = open("/content/ollama.log", "w")
process = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log)

# Esperar hasta que el servidor responda (máx. 60 s)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=1)
        print("[+] Ollama está listo")
        break
    except Exception:
        time.sleep(1)
else:
    print("[!] Ollama no respondió; revisa /content/ollama.log")

!ollama list

In [ ]:
# 4. Descargar los modelos para el Paper
# Puedes comentar o descomentar según la VRAM disponible (T4: 16GB, A100: 40GB)
models_to_pull = [
    "llama3.2:3b",       # 3.2B (Edge)
    "llama3.1:8b",       # 8.0B (Baseline de facto)
    "qwen2.5:7b",        # 7.6B (Multilingüe denso)
    "qwen2.5:14b",       # 14.7B (Sweet spot para A100/L4)
    "aya-expanse:8b",    # 8.0B (Especialista en lenguas originarias)
    # "deepseek-r1:8b",  # 8.0B (Paradigma de razonamiento CoT)
    # "qwen2.5:32b",     # 32B (Solo en A100 con 40GB VRAM)
]

for m in models_to_pull:
    print(f"[*] Descargando modelo: {m}...")
    !ollama pull {m}

!ollama list

In [ ]:
# 5. Prueba de humo: Traducir una oración compleja con andamiaje compacto (-L)
test_sentence = "Allin yuyaymanaywan rimanakuspallanchisraqmi, paqtataq kay sasachakuyninchiskunata allichasunchis churanakuymanta ayqenanchispaq."
!python3 bin/qtr translate "{test_sentence}" --provider ollama --model qwen2.5:7b -L

In [ ]:
# 6. Ejecutar el Benchmark Multi-Modelo Académico
# Configura --start y --end según la cantidad de oraciones que quieras evaluar (ej. 20 o las 100 del dataset)
eval_models = "llama3.2:3b,llama3.1:8b,qwen2.5:7b,qwen2.5:14b,aya-expanse:8b"
!python3 benchmark/multi_model_runner.py \n    --models "{eval_models}" \n    --start 0 \n    --end 25 \n    --output-dir data/benchmark_results

In [ ]:
# 7. Visualizar la Tabla Comparativa Lado a Lado (Con Herramienta vs Sin Herramienta)
import pandas as pd
from IPython.display import display, HTML

csv_path = "data/benchmark_results/translations_comparison_table.csv"
df = pd.read_csv(csv_path)
print(f"Total de oraciones evaluadas: {len(df)}")
display(df.head(10))

In [ ]:
# 8. Ver las Tablas en formato LaTeX listas para el Paper
with open("docs/paper_tables.tex", "r") as f:
    latex_code = f.read()

print(latex_code)

In [ ]:
# 9. Descargar artefactos generados como archivo ZIP
from google.colab import files
!zip -r quechua_academic_benchmark_results.zip data/benchmark_results/ docs/paper_tables.tex
files.download("quechua_academic_benchmark_results.zip")